In [1]:
import pandas as pd

In [2]:
ais_data = pd.read_csv('ais_sample_10000.csv')
ais_data.head()

,mmsi,base_date_time,longitude,latitude,sog,cog,heading,vessel_name,imo,call_sign,vessel_type,status,length,width,draft,cargo,transceiver
0,367793030,2025-01-08 00:00:00,-122.40506,47.68588,4.6,155.5,NaN,WN1622SL,NaN,WDJ5962,37.0,NaN,10.0,3.0,NaN,NaN,B
1,338160209,2025-01-08 00:00:00,-119.69199,34.40719,0.0,106.3,NaN,WESTERLY,NaN,NaN,36.0,NaN,11.0,4.0,NaN,NaN,B
2,266283000,2025-01-08 00:00:01,-74.24126,38.41834,15.5,187.9,190.0,OBERON,IMO9377509,SKJF,70.0,0.0,237.0,32.0,9.1,70.0,A
3,368013620,2025-01-08 00:00:10,-74.04712,40.10181,0.0,252.3,NaN,MARKET PRICE,NaN,WDJ8153,30.0,NaN,17.0,6.0,NaN,NaN,B
4,368144150,2025-01-08 00:00:09,-119.22453,34.16235,0.0,169.6,NaN,SILENT LADY,NaN,WDL5759,37.0,NaN,12.0,5.0,NaN,NaN,B


In [3]:



# Load AIS data (example CSV with vessel details and positions)
ais_data = pd.read_csv('ais_sample_10000.csv')

# Convert base_date_time to datetime format for time-based sorting
ais_data['base_date_time'] = pd.to_datetime(ais_data['base_date_time'])

# Sort by mmsi and base_date_time to analyze each vessel's movement sequentially
ais_data = ais_data.sort_values(['mmsi', 'base_date_time'])

# Detect anomalies in AIS data based on speed and course changes
# Detect anomalies in AIS data based on relative speed changes and course changes
def detect_anomalies(df):
    anomalies = []
    unique_vessels = df['mmsi'].unique()

    for vessel in unique_vessels:
        vessel_data = df[df['mmsi'] == vessel].reset_index(drop=True)

        for i in range(1, len(vessel_data)):
            previous_speed = vessel_data['sog'].iloc[i - 1]
            current_speed = vessel_data['sog'].iloc[i]
            speed_change = abs(current_speed - previous_speed)
            course_change = abs(vessel_data['cog'].iloc[i] - vessel_data['cog'].iloc[i - 1])

            # Detect speed anomalies based on a 50% change relative to the previous speed
            if previous_speed > 0 and speed_change > 0.3 * previous_speed:
                anomalies.append(vessel_data.iloc[i])
            # Detect course anomalies
            elif previous_speed >= 30 and course_change > 30:
                anomalies.append(vessel_data.iloc[i])

    return pd.DataFrame(anomalies)

# Detect anomalies
anomalies_df = detect_anomalies(ais_data)
print("Anomalies detected:")
print(anomalies_df)



Anomalies detected:
         mmsi      base_date_time  longitude  latitude  sog    cog  heading  \
1   257560000 2025-01-08 00:01:34  -81.54645  30.40578  0.4   40.0     18.0   
1   258009000 2025-01-08 00:00:00  -89.99663  29.94090  0.0  353.1    353.0   
1   303398000 2025-01-08 00:01:08 -122.22450  47.98443  0.7  151.0    157.0   
2   310034000 2025-01-08 00:03:10  -64.57621  18.47081  0.0  130.0    127.0   
1   311036900 2025-01-08 00:01:23  -88.09516  30.07468  0.0  199.0    334.0   
..        ...                 ...        ...       ...  ...    ...      ...   
1   538005550 2025-01-08 00:01:27  -94.67772  29.06223  0.2   29.5     50.0   
1   538006053 2025-01-08 00:01:24 -123.41033  37.17197  1.4  134.5     43.0   
1   538072221 2025-01-08 00:01:52  -66.08995  18.45987  1.4  105.0    274.0   
1   636023386 2025-01-08 00:01:17  -96.86043  27.78207  1.9  341.6    350.0   
1   671317100 2025-01-08 00:01:31  -64.96113  18.33008  0.0    NaN     25.0   

          vessel_name         i

In [6]:
import ee
import geemap

# Trigger the authentication
ee.Authenticate(auth_mode='localhost')

# Initialize the Earth Engine API
ee.Initialize(project='gen-lang-client-0814897918')


Successfully saved authorization token.


In [7]:
def plot_histogram(image, region, title):
    try:
        array = image.reduceRegion(
            reducer=ee.Reducer.toList(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        ).get('VV')

        # Check if array is empty
        if array is None:
            print("No VV data available for histogram plot.")
            return

        array = np.array(array.getInfo())
        plt.hist(array, bins=50, color='blue', alpha=0.7)
        plt.title(title)
        plt.xlabel('VV (dB)')
        plt.ylabel('Frequency')
        plt.grid()
        plt.show()
    except Exception as e:
        print(f"Error plotting histogram: {e}")

In [8]:
from datetime import datetime

# Function to parse date from BaseDateTime
def extract_date(base_date_time):
    # Convert to string if it’s a Timestamp
    if isinstance(base_date_time, pd.Timestamp):
        base_date_time = base_date_time.strftime('%Y-%m-%dT%H:%M:%S')
    else:
        base_date_time = str(base_date_time)

    # Parse the date
    date_time_obj = datetime.strptime(base_date_time, '%Y-%m-%dT%H:%M:%S')
    return date_time_obj.strftime('%Y-%m-%d')


In [9]:
def preprocess_image_for_model(image_np):
    """ Preprocess the NumPy array image to match model input requirements """
    # Resize the image to (256, 256)
    resized_image = cv2.resize(image_np, (256, 256))
    
    # Check the number of channels
    if resized_image.shape[-1] == 2:
        # Convert 2-channel input to 3-channel by duplicating one of the channels
        resized_image = np.repeat(resized_image[:, :, :1], 3, axis=-1)
    elif resized_image.shape[-1] == 1:
        # Convert 1-channel input to 3-channel
        resized_image = np.repeat(resized_image, 3, axis=-1)
    elif resized_image.shape[-1] != 3:
        raise ValueError(f"Unexpected number of channels: {resized_image.shape[-1]}")
    
    # Add a batch dimension
    return np.expand_dims(resized_image, axis=0)


In [10]:
import tensorflow as tf

# Define jaccard_coef if not already defined
def jaccard_coef(y_true, y_pred):
    y_true_flatten = tf.reshape(y_true, [-1])
    y_pred_flatten = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_flatten * y_pred_flatten)
    sum_ = tf.reduce_sum(y_true_flatten) + tf.reduce_sum(y_pred_flatten)
    smooth = 1e-6  # To avoid division by zero
    jac = (intersection + smooth) / (sum_ - intersection + smooth)
    return jac


In [15]:
from tensorflow.keras.models import load_model
import os

# Ensure segmentation_models uses TensorFlow backend
os.environ["SM_FRAMEWORK"] = "tf.keras"
import segmentation_models as sm

# Load the models

unet_model = load_model(r"C:\Users\MAYANK\Desktop\sih-2026\SpillTrace-SIH26\ml\unet_model.h5")



In [16]:
def preprocess_image_for_model(image_np):
    """ Preprocess the NumPy array image to match model input requirements """
    # Remove any extra dimensions if present (e.g., shape (256, 256, 1, 6) -> (256, 256, 2))
    if image_np.ndim == 4:
        image_np = image_np[:, :, :, 0]

    # Resize to 256x256
    resized_image = cv2.resize(image_np, (256, 256))

    # Ensure the image has three channels (duplicate one if necessary)
    if resized_image.shape[-1] == 2:
        resized_image = np.repeat(resized_image[:, :, :1], 3, axis=-1)
    elif resized_image.shape[-1] == 1:
        resized_image = np.repeat(resized_image, 3, axis=-1)

    # Add batch dimension
    return np.expand_dims(resized_image, axis=0)

In [26]:
def confirm_oil_spill(unet_model, image):
    """ 
    Confirm oil spill detection using Hybrid, UNet, and DeepLabV3+ models. 
    Requires agreement among all models.
    """
    processed_image = preprocess_image_for_model(image)

    processed_image = preprocess_image_for_model(image)
        
        # Get predictions from both models
    unet_prediction = unet_model.predict(processed_image)
    # deeplab_prediction = deeplab_model.predict(processed_image)
    # hybrid_prediction = hybrid_model.predict(processed_image)

    # Assuming '1' corresponds to oil spill for both models
    unet_result = np.argmax(unet_prediction) == 1
    # deeplab_result = np.argmax(deeplab_prediction) == 1
    # hybrid_result = np.argmax(hybrid_prediction) == 1

    # Confirm only if both models agree
    #return unet_result 

    # Confirm only if all models agree
    return unet_result

In [27]:
from tensorflow.keras.models import load_model
import geemap
import ee
import numpy as np
import cv2
from IPython.display import display

def process_anomaly(row):
    lat = row['latitude']
    lon = row['longitude']
    base_date_time = row['base_date_time']
    date = extract_date(base_date_time)
    anomaly_date = ee.Date(date)
    print(f"Processing anomaly at {lat}, {lon} on {date}")

    roi = ee.Geometry.Point(lon, lat).buffer(10000).bounds()
    detected_spills = []

    # Sentinel-1 collection
    sen1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterDate(anomaly_date, anomaly_date.advance(1, 'day')) \
        .filterBounds(roi) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    if sen1.size().getInfo() == 0:
        print("No Sentinel-1 VV or VH images available. Trying Sentinel-2.")
        sen1 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
            .filterDate(anomaly_date, anomaly_date.advance(1, 'day')) \
            .filterBounds(roi) \
            .filter(ee.Filter.eq('CLOUDY_PIXEL_PERCENTAGE', 0))

        if sen1.size().getInfo() == 0:
            print("No Sentinel-2 images available for this date and location.")
            return None

        sen1_image = sen1.select(['B4', 'B8']).mosaic()
    else:
        sen1_image = sen1.select(['VV', 'VH']).mosaic()

    despeckled = sen1_image.focal_mean(100, 'square', 'meters')
    vv_dark = despeckled.select('VV').lt(-22)
    vh_dark = despeckled.select('VH').lt(-18)
    oil_spill_detection = vv_dark.And(vh_dark)

    mask = oil_spill_detection.updateMask(oil_spill_detection)
    area = mask.multiply(ee.Image.pixelArea().divide(1e6))
    oil_spill_area = ee.Number(
        area.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=roi,
            scale=100
        ).values().get(0)
    ).getInfo()

    if oil_spill_area > 0:
        print(f"Detected Oil Spill Area (sq. km): {oil_spill_area}")

        despeckled_np = geemap.ee_to_numpy(despeckled, bands=['VV', 'VH'], region=roi, scale=100)

        # Ensure image preprocessing for models
        #processed_image = preprocess_image_for_model(despeckled_np)

        # Confirm using all three models
        if confirm_oil_spill(unet_model,despeckled_np):
            detected_spills.append(oil_spill_area)

            # oil_spill_vis_params = {
            #     'min': 0,
            #     'max': 1,
            #     'palette': ['blue', 'cyan', 'yellow', 'red']
            # }

            # Visualization
            Map = geemap.Map()
            Map.centerObject(roi)
            Map.addLayer(sen1_image, {'bands': ['VV', 'VH'], 'min': -30, 'max': 0}, f'Sentinel-1 Image {anomaly_date.getInfo()}')
            Map.addLayer(despeckled.clip(roi), {}, 'Despeckled Image', False)
            Map.addLayer(oil_spill_detection.clip(roi), {}, 'Oil Spill Detection', False)
            oil_spill_vector = mask.reduceToVectors(
                geometry=roi,
                scale=100
            )
            Map.addLayer(oil_spill_vector, {}, 'Oil Spill Vector', False)

    if detected_spills:
        print(f"Oil spills detected over the days: {detected_spills}")
        Map.addLayerControl()
        return Map
    else:
        print("No significant oil spills detected across the checked dates.")
        return None


In [29]:
# Iterate over each anomaly in the DataFrame and process
for index, row in anomalies_df.head(10).iterrows():
    result_map = process_anomaly(row)
    if result_map:
        display(result_map)  # This displays the map in interactive environments

Processing anomaly at 30.40578, -81.54645 on 2025-01-08
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 29.9409, -89.99663 on 2025-01-08
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 47.98443, -122.2245 on 2025-01-08
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 18.47081, -64.57621 on 2025-01-08
Detected Oil Spill Area (sq. km): 117.49171283523427
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
No significant oil spills detected across the checked dates.
Processing anomaly at 30.07468, -88.09516 on 2025-01-08
No Sentinel-1 VV or VH images available. Trying Sentinel-2.
No Sentinel-2 images available for this date and location.
Processing anomaly at 29.93947, -90.14193 on 2025-01-08
No Sentinel-1 VV or VH images availab